<div style="background:linear-gradient(90deg,#C8102E 0%,#7A0019 100%); color:white; padding:22px 28px; border-radius:8px">
  <div style="font-size:0.9em; letter-spacing:2px; opacity:0.85">NOTEBOOK 07 · LLM EVALUATION · COMPLETED</div>
  <div style="font-size:2.0em; font-weight:700; margin-top:4px">📊 Evaluating the extraction: prompts, models, traces, and judges</div>
  <div style="font-size:1.1em; margin-top:8px; max-width:880px; opacity:0.95">
    Score the biomarker extraction against ground truth two ways: a transparent hand-computed
    accuracy, then the managed <code>mlflow.genai.evaluate()</code> path with per-row traces, an
    LLM-as-judge, and a custom metric.
  </div>
</div>

## Why evaluate an LLM extraction step at all?

In nb 04, `ai_query` *worked* on a spot check, but "looks right on a few rows" is not how you ship
something a trial-eligibility decision leans on. You'd never ship an ML model without a test set and
a number. **The same discipline applies to an LLM prompt.**

This notebook has two halves:
1. **Sections 1 to 5**: a transparent experiment (2 prompts × 2 models) where we compute accuracy
   ourselves and log each run to MLflow, so every number on screen is explainable.
2. **Sections 6 to 8**: the managed **`mlflow.genai.evaluate()`** path, which auto-captures a
   **trace** per row, runs an **LLM-as-judge**, and lets us plug in a **custom metric**.

In [0]:
%pip install -U "mlflow>=3.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 614.2/614.2 kB 12.2 MB/s eta 0:00:00
  Attempting uninstall: wcwidth
    Found existing installation: wcwidth 0.2.5
    Not uninstalling wcwidth at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-6018fa6d-4783-4c29-aae5-f55173dac0ee
    Can't uninstall 'wcwidth'. No files were found to uninstall.
  Attempting uninstall: blinker
    Found existing installation: blinker 1.7.0
    Not uninstalling blinker at /usr/lib/python3/dist-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-6018fa6d-4783-4c29-aae5-f55173dac0ee
    Can't uninstall 'blinker'. No files 

In [0]:
dbutils.library.restartPython()

In [0]:
%run ./_config

# ⚙️ Configuration: Clinical Trial Pre-Screening (PRE-BUILT)

<div style="background:#f4f6f9; border-left:6px solid #C8102E; padding:14px 18px; border-radius:4px; font-size:0.95em">
This is the <b>companion config notebook</b>. It is <b>pre-built; you do not edit it</b>.
Every other notebook starts with <code>%run ./_config</code> so they all share one
catalog / schema / warehouse and the same read-only OMOP source.<br>
Just set the widgets at the top of <code>00_START_HERE</code> (matching your
bundle's <code>client_catalog</code> / <code>client_schema</code> / <code>warehouse_id</code>
/ <code>source_schema</code>) and re-run.
</div>

Everything here is Unity-Catalog-scoped (no hive_metastore) and reads from widgets, no
hardcoded secrets.

ℹ️ Not creating catalog clinops_catalog (likely pre-provisioned / no CREATE CATALOG): (com.databricks.sql.managedcatalog.acl.UnauthorizedAccessException) PERMISSION_D
✅ Writing to clinops_catalog.clinops_ml
   Reading read-only OMOP source from clinops_catalog.clinops_foundation
   SQL Warehouse: 0123456789abcdef


Helpers ready: fqn(), src(), show_md(), LLM_FAST, LLM_STRONG


## 1️⃣ Build the ground-truth eval set (PRE-BUILT)

One row per patient 1 to 180 carrying **both** the note text *and* the structured HER2/ER/PR labels.
Patients 1 to 180 are the **both-agree** cohort: the structured value is the gold label, the note is
what the model reads.

In [0]:
%sql
CREATE OR REPLACE TABLE eval_biomarker_goldset
COMMENT 'Both-agree cohort (person 1-180): pathology note text + structured HER2/ER/PR gold labels'
AS
WITH notes AS (
  SELECT person_id, MAX(note_text) AS note_text
  FROM note
  WHERE note_source_value = 'PATHOLOGY_REPORT' AND person_id BETWEEN 1 AND 180
  GROUP BY person_id
)
SELECT n.person_id, n.note_text,
  g.her2_status AS gold_her2, g.er_status AS gold_er, g.pr_status AS gold_pr
FROM notes n
JOIN silver_biomarker_profile g ON n.person_id = g.person_id
WHERE g.her2_status IS NOT NULL OR g.er_status IS NOT NULL OR g.pr_status IS NOT NULL;

num_affected_rows,num_inserted_rows


## 2️⃣ Define the two prompts (COMPLETED)

Both return the **same strict values** (`Positive`/`Negative`/`Unknown`). The only difference is
how much clinical guidance each gives about reading IHC / FISH results. That contrast is the lesson.

In [0]:
PROMPT_V1 = (
    "Extract the HER2, ER (estrogen receptor), and PR (progesterone receptor) status from this breast "
    "cancer pathology report. Answer with exactly one of Positive, Negative, or Unknown for each."
)

PROMPT_V2 = (
    "You are a breast pathology expert. Extract HER2, ER (estrogen receptor), and PR (progesterone "
    "receptor) status from the pathology report using these rules. "
    "HER2: IHC 3+ OR FISH-amplified => Positive; IHC 0 or 1+ OR FISH not amplified => Negative; "
    "IHC 2+ without confirming FISH, or otherwise equivocal => Unknown. "
    "ER and PR: >= 1% nuclear staining => Positive; < 1% => Negative; not reported => Unknown. "
    "Answer with exactly one of Positive, Negative, or Unknown for each marker."
)

PROMPTS = {"v1_terse": PROMPT_V1, "v2_careful": PROMPT_V2}
MODELS  = [LLM_FAST, LLM_STRONG]   # databricks-claude-haiku-4-5, databricks-claude-sonnet-4-6
print("Prompts:", list(PROMPTS.keys()))
print("Models :", MODELS)

Prompts: ['v1_terse', 'v2_careful']
Models : ['databricks-claude-haiku-4-5', 'databricks-claude-sonnet-4-6']


## 3️⃣ Run all four configurations and log to MLflow (COMPLETED)

For each (prompt, model) pair: one `ai_query` pass, compute per-marker accuracy vs gold, and log
**one MLflow run** so the Experiments UI lines the four up. To keep the live run quick we score a
**60-patient slice** of the goldset (all 30 equivocal hard cases, person 61-90, plus 30 clear ones,
person 1-30). Concentrating the hard cases makes the terse-vs-careful gap easy to see; drop the
`WHERE` clause below to score the full 180 both-agree set.

In [0]:
import mlflow
from pyspark.sql import functions as F

mlflow.set_experiment(f"/Users/{spark.sql('SELECT current_user()').first()[0]}/clinops_biomarker_eval")
GOLDSET = fqn("eval_biomarker_goldset")


def run_config(model: str, prompt_name: str, prompt_text: str):
    """Score one (model, prompt) config on the goldset and log an MLflow run."""
    safe_prompt = prompt_text.replace("'", "''")  # SQL-escape single quotes

    # COMPLETED TODO #1, the scoring query. One ai_query pass over the goldset. We use the same
    # result-wrapper responseFormat as nb 04 (the DDL allows one top-level field) and read the nested
    # struct fields directly, aliasing them pred_her2 / pred_er / pred_pr.
    # ai_query with responseFormat returns a JSON STRING on this runtime, so we parse it with
    # from_json using the FLAT inner schema (same proven pattern as nb 04), then read the fields.
    preds = spark.sql(f"""
        SELECT person_id, note_text, gold_her2, gold_er, gold_pr,
               x.her2_status AS pred_her2,
               x.er_status   AS pred_er,
               x.pr_status   AS pred_pr
        FROM (
          SELECT person_id, note_text, gold_her2, gold_er, gold_pr,
            from_json(
              ai_query(
                '{model}',
                '{safe_prompt}' || '\\n\\nReport:\\n' || note_text,
                responseFormat => 'STRUCT<result:STRUCT<her2_status:STRING, er_status:STRING, pr_status:STRING>>'
              ),
              'STRUCT<her2_status:STRING, er_status:STRING, pr_status:STRING>'
            ) AS x
          FROM {GOLDSET}
          WHERE person_id BETWEEN 1 AND 30 OR person_id BETWEEN 61 AND 90
        )
    """)

    # Per-marker accuracy, PRE-BUILT (only scores rows where a gold label exists).
    def acc(gold_col, pred_col):
        return (
            F.sum(F.when(F.col(gold_col).isNotNull() & (F.col(gold_col) == F.col(pred_col)), 1).otherwise(0))
            / F.sum(F.when(F.col(gold_col).isNotNull(), 1).otherwise(0))
        )

    scored = preds.agg(
        acc("gold_her2", "pred_her2").alias("her2_acc"),
        acc("gold_er",   "pred_er").alias("er_acc"),
        acc("gold_pr",   "pred_pr").alias("pr_acc"),
    ).first()
    her2 = float(scored["her2_acc"] or 0.0)
    er   = float(scored["er_acc"]   or 0.0)
    pr   = float(scored["pr_acc"]   or 0.0)
    overall = (her2 + er + pr) / 3.0

    # COMPLETED TODO #2, log this configuration as ONE MLflow run. Params + metrics per run are what
    # make the four configs comparable in the Experiments UI.
    with mlflow.start_run(run_name=f"{model}__{prompt_name}"):
        mlflow.log_params({"model": model, "prompt_name": prompt_name, "prompt_text": prompt_text})
        mlflow.log_metrics({"her2_acc": her2, "er_acc": er, "pr_acc": pr, "overall_acc": overall})

    return {
        "model": model, "prompt": prompt_name,
        "her2_acc": round(her2, 4), "er_acc": round(er, 4),
        "pr_acc": round(pr, 4), "overall_acc": round(overall, 4),
        "preds": preds,
    }


results = []
for model in MODELS:
    for prompt_name, prompt_text in PROMPTS.items():
        print(f"▶ scoring  {model}  ×  {prompt_name} …")
        results.append(run_config(model, prompt_name, prompt_text))
print(f"\n✅ {len(results)} configurations scored and logged to MLflow.")

2026/07/13 18:18:49 INFO mlflow.tracking.fluent: Experiment with name '/Users/ml-team@example.com/clinops_biomarker_eval' does not exist. Creating a new experiment.
If you are using MLflow Tracing, consider storing your traces in Unity Catalog for unlimited storage (no 100,000 trace limit), fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/trace-unity-catalog


▶ scoring  databricks-claude-haiku-4-5  ×  v1_terse …
▶ scoring  databricks-claude-haiku-4-5  ×  v2_careful …
▶ scoring  databricks-claude-sonnet-4-6  ×  v1_terse …
▶ scoring  databricks-claude-sonnet-4-6  ×  v2_careful …

✅ 4 configurations scored and logged to MLflow.


## 4️⃣ Compare the four runs and pick a winner (PRE-BUILT)

In [0]:
comparison = spark.createDataFrame(
    [{k: r[k] for k in ("model", "prompt", "her2_acc", "er_acc", "pr_acc", "overall_acc")} for r in results]
).orderBy(F.col("overall_acc").desc())
display(comparison)

er_acc,her2_acc,model,overall_acc,pr_acc,prompt
1.0,1.0,databricks-claude-sonnet-4-6,1.0,1.0,v2_careful
1.0,0.85,databricks-claude-haiku-4-5,0.95,1.0,v2_careful
0.85,1.0,databricks-claude-sonnet-4-6,0.95,1.0,v1_terse
0.75,0.7,databricks-claude-haiku-4-5,0.8167,1.0,v1_terse


In [0]:
best = max(results, key=lambda r: r["overall_acc"])
show_md(f"""
<div style="background:#E8F5E9; border-left:6px solid #2E7D32; padding:12px 16px; border-radius:4px">
🏆 <b>Winner:</b> <code>{best['model']}</code> with prompt <b>{best['prompt']}</b>,
overall accuracy <b>{best['overall_acc']*100:.1f}%</b>
(HER2 {best['her2_acc']*100:.1f}% · ER {best['er_acc']*100:.1f}% · PR {best['pr_acc']*100:.1f}%).
</div>
""")

🏆 Winner: databricks-claude-sonnet-4-6 with prompt v2_careful ,
overall accuracy 100.0% 
(HER2 100.0% · ER 100.0% · PR 100.0%).

## 5️⃣ Error patterns: where the winner disagreed with the gold label (PRE-BUILT)

In [0]:
winner_preds = best["preds"].withColumn("note_snippet", F.substring("note_text", 1, 220))
errors = (
    winner_preds
    .select(
        "person_id", "note_snippet",
        F.explode(F.array(
            F.struct(F.lit("HER2").alias("marker"), F.col("gold_her2").alias("actual"), F.col("pred_her2").alias("predicted")),
            F.struct(F.lit("ER").alias("marker"),   F.col("gold_er").alias("actual"),   F.col("pred_er").alias("predicted")),
            F.struct(F.lit("PR").alias("marker"),   F.col("gold_pr").alias("actual"),   F.col("pred_pr").alias("predicted")),
        )).alias("m"),
    )
    .where(F.col("m.actual").isNotNull() & (F.col("m.actual") != F.col("m.predicted")))
    .select("person_id", "m.marker", "m.predicted", "m.actual", "note_snippet")
    .orderBy("marker", "person_id")
)
print(f"Disagreements for {best['model']} × {best['prompt']}:")
display(errors)

Disagreements for databricks-claude-sonnet-4-6 × v2_careful:


person_id,marker,predicted,actual,note_snippet


<div style="background:#FFEBEE; border-left:6px solid #C62828; padding:12px 16px; border-radius:4px">
<b>How to read the misses.</b> They should cluster on the genuinely hard cases: HER2 <b>IHC 2+</b>
(equivocal) and borderline ER. A miss that lands on <code>Unknown</code> is far safer than a
confident wrong call. The careful <code>v2</code> prompt was written to push ambiguous cases
toward <code>Unknown</code>.
</div>

# ═══ The managed path: `mlflow.genai.evaluate()` ═══

The sections above computed accuracy by hand so every number is transparent. In practice you'd
often reach for **`mlflow.genai.evaluate()`**, which gives you three things the hand path does not:

<div style="display:flex; gap:14px; flex-wrap:wrap; margin-top:8px">
  <div style="flex:1; min-width:220px; background:#E3F2FD; border-radius:6px; padding:14px">
    <b>🔍 Traces</b><br>A full, inspectable record of each model call (input, output, latency) in
    the <b>Traces</b> tab, one per row. Click any patient to see exactly what the model saw and said.
  </div>
  <div style="flex:1; min-width:220px; background:#F3E5F5; border-radius:6px; padding:14px">
    <b>⚖️ LLM-as-judge</b><br>A built-in scorer that uses an LLM to grade each response against a
    plain-language rule, for quality dimensions you cannot check with an equality test.
  </div>
  <div style="flex:1; min-width:220px; background:#E8F5E9; border-radius:6px; padding:14px">
    <b>🧮 Custom metrics</b><br>Your own Python scorer (here, exact-match against the gold label),
    evaluated per row and aggregated automatically alongside the judge.
  </div>
</div>

## 6️⃣ A traced prediction function, aimed at the hard cases

`mlflow.genai.evaluate()` calls a `predict_fn` once per row and captures a **trace** of each call.
We decorate ours with `@mlflow.trace` and reuse the proven `ai_query` extraction.

<div style="background:#FFF8E1; border-left:6px solid #F2A900; padding:12px 16px; border-radius:4px">
<b>Where the learning is.</b> We point the eval at the <b>hard-case band</b> (person 61-90): notes
written with equivocal-but-resolvable phrasing (HER2 IHC 2+ with a reflex FISH ratio,
ER-low-positive), where the structured value is still the definite gold label. This is where a
terse prompt and a careful prompt <i>disagree</i>, so the traces show real, instructive misses
(not a wall of 100%). We build the extractor as a factory so we can evaluate two prompts and
compare them in the managed harness.
</div>

In [0]:
import mlflow

# We evaluate on the HARD-CASE band (person 61-90): both-agree patients whose STRUCTURED value is
# the definite gold label, but whose pathology note is written equivocally (HER2 IHC 2+ with a
# reflex FISH ratio, ER-low-positive). That is exactly where a terse prompt and a careful prompt
# diverge, so the traces and metrics have something to show. We add a few clear cases (person 1-5)
# so the slice is not all-hard. Kept small so the two eval runs below stay snappy in a live demo.
gold_pdf = spark.sql(f"""
    SELECT person_id, note_text, gold_her2, gold_er, gold_pr,
           CASE WHEN person_id BETWEEN 61 AND 90 THEN 'hard' ELSE 'clear' END AS case_type
    FROM {GOLDSET}
    WHERE note_text IS NOT NULL
      AND (person_id BETWEEN 61 AND 80 OR person_id BETWEEN 1 AND 5)
    ORDER BY person_id
""").toPandas()
print(f"Eval rows: {len(gold_pdf)}  "
      f"(hard: {(gold_pdf.case_type == 'hard').sum()}, clear: {(gold_pdf.case_type == 'clear').sum()})")

# MLflow's eval dataset format: one dict per row with "inputs" and "expectations".
eval_data = [
    {
        "inputs": {"note_text": r.note_text},
        "expectations": {"her2": r.gold_her2, "er": r.gold_er, "pr": r.gold_pr},
    }
    for r in gold_pdf.itertuples()
]


def make_extractor(prompt_text: str, model: str = LLM_FAST):
    """Build a traced predict_fn bound to a specific prompt and model, so we can evaluate more than
    one configuration and compare. @mlflow.trace records each call as a trace MLflow attaches to the
    evaluation run.

    We deliberately default to the FAST model here. The leaderboard above shows the strong model
    scores ~100% on both prompts, so it MASKS the prompt difference; the fast model is where prompt
    quality actually moves the needle, which is the honest lesson: engineer the prompt when you are
    optimizing a smaller, cheaper model, and the equivocal cases are where that work pays off."""
    safe_prompt = prompt_text.replace("'", "''")

    @mlflow.trace
    def _extract(note_text: str) -> dict:
        safe_note = (note_text or "").replace("'", "''")
        row = spark.sql(f"""
            SELECT x.her2_status AS her2, x.er_status AS er, x.pr_status AS pr
            FROM (
              SELECT from_json(
                ai_query(
                  '{model}',
                  '{safe_prompt}' || '\\n\\nReport:\\n' || '{safe_note}',
                  responseFormat => 'STRUCT<result:STRUCT<her2_status:STRING, er_status:STRING, pr_status:STRING>>'
                ),
                'STRUCT<her2_status:STRING, er_status:STRING, pr_status:STRING>'
              ) AS x
            )
        """).first()
        return {"her2": row["her2"], "er": row["er"], "pr": row["pr"]}

    return _extract


# Smoke-test the careful extractor on one HARD note (should resolve the equivocal FISH correctly).
_hard_note = gold_pdf[gold_pdf.case_type == "hard"].iloc[0]["note_text"]
print(make_extractor(PROMPT_V2)(_hard_note))

Eval rows: 25  (hard: 20, clear: 5)
{'her2': 'Negative', 'er': 'Negative', 'pr': 'Negative'}


Trace(trace_id=tr-1da3ea627f292a0ce123e8d525b13785)

## 7️⃣ Scorers: a custom metric AND an LLM-as-judge (COMPLETED)

- **`her2_exact_match`** and **`biomarker_agreement`** are **custom metrics**: plain Python
  functions decorated with `@scorer`, evaluated per row against the gold `expectations`.
- **`valid_status_values`** is a built-in **`Guidelines`** scorer: an **LLM-as-judge** that reads
  each response and grades it against a plain-language rule. This catches quality issues (a stray
  value like "equivocal" instead of "Unknown") that an equality test would miss.

In [0]:
from mlflow.genai.scorers import scorer, Guidelines


@scorer
def her2_exact_match(outputs: dict, expectations: dict) -> bool:
    """Custom metric: did the predicted HER2 exactly match the gold HER2?"""
    return outputs.get("her2") == expectations.get("her2")


@scorer
def biomarker_agreement(outputs: dict, expectations: dict):
    """Custom metric: fraction of the three markers (that have a gold label) the model got right."""
    keys = ["her2", "er", "pr"]
    graded = [k for k in keys if expectations.get(k) is not None]
    if not graded:
        return None
    correct = sum(1 for k in graded if outputs.get(k) == expectations.get(k))
    return correct / len(graded)


# LLM-as-judge: grades each response against a plain-language rule. Uses a Databricks-hosted judge
# model by default; if your workspace has no judge model configured, drop this scorer from the list.
valid_status_values = Guidelines(
    name="valid_status_values",
    guidelines=(
        "The response must report her2, er, and pr. Each of those three values must be exactly one of "
        "'Positive', 'Negative', or 'Unknown'. Any other wording (for example 'equivocal', 'amplified', "
        "or a numeric score) fails."
    ),
)

print("Scorers ready: her2_exact_match (custom), biomarker_agreement (custom), valid_status_values (LLM judge).")

Scorers ready: her2_exact_match (custom), biomarker_agreement (custom), valid_status_values (LLM judge).


## 8️⃣ Run `mlflow.genai.evaluate()` for both prompts (COMPLETED)

We evaluate **two configurations** on the hard slice, the terse prompt and the careful prompt,
each as its own MLflow run, both on the **fast model**. Every run applies all three scorers and
records a trace per row. We use the fast model on purpose: the leaderboard above shows the strong
model scores ~100% on both prompts, so it **hides** the prompt difference. The fast model is where
prompt quality moves the number, so the **careful prompt scores higher** on `her2_exact_match` and
`biomarker_agreement`, because it resolves the equivocal FISH and ER-low-positive notes the terse
prompt slips on. Open each run in the **Experiments** UI, then the **Traces** tab, to click into a
single patient's call.

In [0]:
import mlflow

configs = {"v1_terse": PROMPT_V1, "v2_careful": PROMPT_V2}
genai_results = {}
for name, prompt_text in configs.items():
    with mlflow.start_run(run_name=f"genai_evaluate__{name}"):
        res = mlflow.genai.evaluate(
            data=eval_data,
            predict_fn=make_extractor(prompt_text),
            scorers=[her2_exact_match, biomarker_agreement, valid_status_values],
        )
        genai_results[name] = res.metrics

# Render the contrast as a table (not console text) so it reads on a big screen. One column per
# prompt, one row per metric; the match metrics should be higher for the careful prompt.
import pandas as pd

_metric_names = sorted({k for m in genai_results.values() for k in m.keys()})
metrics_comparison = pd.DataFrame(
    {name: [genai_results[name].get(k) for k in _metric_names] for name in configs},
    index=_metric_names,
).reset_index().rename(columns={"index": "metric"})
print("Managed-eval metrics, terse vs careful (higher is better on the match metrics):")
display(metrics_comparison)

2026/07/13 18:21:12 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/07/13 18:21:12 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/25 [Elapsed: 00:00, Remaining: ?]

2026/07/13 18:21:31 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/25 [Elapsed: 00:00, Remaining: ?]

Managed-eval metrics, terse vs careful (higher is better on the match metrics):


metric,v1_terse,v2_careful
biomarker_agreement/mean,0.7066666666666666,0.92
her2_exact_match/mean,0.52,0.76
valid_status_values/mean,0.48,0.6


In [0]:
# The clearest way to SEE the contrast without opening the Traces UI: line up the two prompts' HER2
# predictions on the equivocal hard-case band, side by side with the gold label, from the section-3
# scoring pass. We use the FAST model, because that is where the prompt difference shows (the strong
# model scores ~100% on both). The rows where terse missed and careful caught it are the equivocal
# IHC 2+/reflex-FISH notes, exactly the cases the careful prompt was written for.
from pyspark.sql import functions as F


def _preds_for(prompt_name, model=LLM_FAST):
    return next(r["preds"] for r in results if r["model"] == model and r["prompt"] == prompt_name)


_terse = _preds_for("v1_terse").select(
    "person_id", "note_text", "gold_her2", F.col("pred_her2").alias("terse_her2")
)
_careful = _preds_for("v2_careful").select("person_id", F.col("pred_her2").alias("careful_her2"))

head_to_head = (
    _terse.join(_careful, "person_id")
    .where(F.col("person_id").between(61, 90))  # the equivocal hard-case band
    .withColumn("terse_ok", F.col("terse_her2") == F.col("gold_her2"))
    .withColumn("careful_ok", F.col("careful_her2") == F.col("gold_her2"))
    .withColumn("note_snippet", F.substring("note_text", 1, 180))
    .select(
        "person_id", "gold_her2", "terse_her2", "terse_ok",
        "careful_her2", "careful_ok", "note_snippet",
    )
    .orderBy("person_id")
)
print("HER2 on the hard-case band, terse vs careful (watch terse_ok=false flip to careful_ok=true):")
display(head_to_head)

HER2 on the hard-case band, terse vs careful (watch terse_ok=false flip to careful_ok=true):


person_id,gold_her2,terse_her2,terse_ok,careful_her2,careful_ok,note_snippet
61,Negative,Positive,false,Negative,true,"PATHOLOGY REPORT — CORE NEEDLE BIOPSY PATH-284673 | 2022-12-05 SPECIMEN: Left breast, stereotactic biopsy INDICATION: Known Left breast carcinoma; re-biopsy for receptor status."
62,Negative,Positive,false,Negative,true,CONSULTATION SURGICAL PATHOLOGY Meridian Cancer Center — Department of Pathology Accession: PATH-388929 Outside MRN: MRN-000062 Date of Consultation: 05/08/2024 SUBMITTED MATERIA
63,Negative,Positive,false,Positive,false,"MERIDIAN CANCER CENTER DEPARTMENT OF PATHOLOGY — SURGICAL PATHOLOGY REPORT Accession No.: PATH-278602 Date Received: December 16, 2024 Date Reported: December 17, 2024 Patient MR"
64,Negative,Positive,false,Negative,true,"PATHOLOGY REPORT — CORE NEEDLE BIOPSY PATH-227476 | 2023-09-03 SPECIMEN: Left breast, core needle biopsy INDICATION: Newly diagnosed Left breast cancer; receptor studies request"
65,Positive,Positive,true,Positive,true,CONSULTATION SURGICAL PATHOLOGY Meridian Cancer Center — Department of Pathology Accession: PATH-388494 Outside MRN: MRN-000065 Date of Consultation: 08/20/2023 SUBMITTED MATERIA
66,Positive,Positive,true,Positive,true,"MERIDIAN CANCER CENTER DEPARTMENT OF PATHOLOGY — SURGICAL PATHOLOGY REPORT Accession No.: PATH-539833 Date Received: March 19, 2024 Date Reported: March 21, 2024 Patient MRN: MRN"
67,Positive,Positive,true,Positive,true,"PATHOLOGY REPORT — CORE NEEDLE BIOPSY PATH-630761 | 2023-07-31 SPECIMEN: Left breast, core needle biopsy INDICATION: Left breast density with suspicious calcifications on mammog"
68,Negative,Positive,false,Negative,true,CONSULTATION SURGICAL PATHOLOGY Meridian Cancer Center — Department of Pathology Accession: PATH-971606 Outside MRN: MRN-000068 Date of Consultation: 04/25/2024 SUBMITTED MATERIA
69,Negative,Positive,false,Negative,true,"MERIDIAN CANCER CENTER DEPARTMENT OF PATHOLOGY — SURGICAL PATHOLOGY REPORT Accession No.: PATH-790446 Date Received: August 11, 2023 Date Reported: August 12, 2023 Patient MRN: M"
70,Positive,Positive,true,Positive,true,"PATHOLOGY REPORT — CORE NEEDLE BIOPSY PATH-961789 | 2023-01-18 SPECIMEN: Left breast, core needle biopsy INDICATION: Newly diagnosed Left breast cancer; receptor studies request"


<div style="background:#E8F5E9; border-left:6px solid #2E7D32; padding:16px 20px; border-radius:6px">
<b>What to show your stakeholders here:</b>
<ol>
<li><b>The contrast</b>: the careful prompt beats the terse one on <code>her2_exact_match</code>
    and <code>biomarker_agreement</code>. Same model, same notes; only the instruction changed.</li>
<li><b>Traces tab</b>: open the terse run, find a hard case (a HER2 IHC 2+ note with a reflex FISH
    ratio around 2.1). You can see the model answered "Unknown" because it stopped at "equivocal".
    Open the same patient in the careful run and it resolved to the right call. That side-by-side is
    the audit trail for an AI-assisted clinical read.</li>
<li><b>The judge column</b>: <code>valid_status_values</code> is graded by an LLM, with a
    rationale per row, not by an equality test.</li>
<li><b>The custom metrics</b>: <code>her2_exact_match</code> and <code>biomarker_agreement</code>
    sit right beside the judge, aggregated over the eval set.</li>
</ol>
The equivocal cases are where the risk lives: a confident wrong call on an ambiguous note is the
thing you must catch. The traces, the judge, and the metrics together are how you catch and measure
it. <b>This is how you ship an LLM step responsibly.</b>
</div>

## ▶️ Next step
### → Open **[08_genie_space_setup]($./08_genie_space_setup)** to put a natural-language layer over the cohort with Genie.